## Imports

In [14]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import zipfile as zipfile

import sys
import pickle
import requests

## Global variables

In [15]:
# URL of datasources
url_mdl                = 'https://files.grouplens.org/datasets/movielens/ml-20m.zip'

url_imdb_namebasics    = 'https://datasets.imdbws.com/name.basics.tsv.gz'
url_imdb_titleakas     = 'https://datasets.imdbws.com/title.akas.tsv.gz'
url_imdb_titlecrew     = 'https://datasets.imdbws.com/title.crew.tsv.gz'
url_imdb_titlebasics   = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_imdb_titleepisode  = 'https://datasets.imdbws.com/title.episode.tsv.gz'
url_imdb_titleprincipals  = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_imdb_titleratings  = 'https://datasets.imdbws.com/title.ratings.tsv.gz'

## Pre-process

In [16]:
# Common films between MDL and IMDB datasets
imdb_mdl_mapp = pd.read_csv('data_processed/imdb_mdl_mapp.csv', dtype={'tconst':object, 'movieId':object, 'primaryTitle':object, 'startYear':int})

# Reference list to limit datasets
list_movies_mdl = imdb_mdl_mapp[(imdb_mdl_mapp['startYear']>2000) & (imdb_mdl_mapp['startYear']<2010)]['movieId'].tolist()

print(imdb_mdl_mapp.info())
imdb_mdl_mapp.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41346 entries, 0 to 41345
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0    41346 non-null  int64 
 1   tconst        41346 non-null  object
 2   movieId       41346 non-null  object
 3   primaryTitle  41346 non-null  object
 4   startYear     41346 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 1.6+ MB
None


,Unnamed: 0,tconst,movieId,primaryTitle,startYear
0,0,tt0000005,95541,Blacksmith Scene,1893
1,1,tt0000008,88674,Edison Kinetoscopic Record of a Sneeze,1894
2,2,tt0000010,120869,Leaving the Factory,1895


In [22]:
# Load movie dataset
movies = pd.read_csv('data_mdl/movies.zip',compression='zip', sep=',')

movies['movieId'] = movies['movieId'].astype('str')

# Limitation of dataset using pre-defined list
# movies = movies[movies['movieId'].isin(list_movies_mdl)]

# Split year from title
movies['year'] = movies['title'].str.extract('.*\((.*)\).*',expand = False)
movies['title'] = movies['title'].apply(lambda x: x[:-7])

# Clean of years values
# dict_replace = {'1975-1979': '1979',
# '2009– ': '2009',
# '2007-':'2007', 
# 'Das Millionenspiel':'1970', 
# 'Bicicleta, cullera, poma':'2010',
# '1983)':'1983'
# }

# movies['year'] = movies['year'].replace(to_replace = dict_replace) 

# Drop of rows with missing year (17 movies)
# movies_without_year = movies[movies['year'].isnull()][['movieId', 'title']]
# movies = movies.dropna(axis=0, how='any', subset=['year'])

# Change 'year' type for int
# movies['year'] = movies['year'].astype('int')

# Clean of genres column
movies['genres'] = movies['genres'].str.replace('|',' ')


print('Nbre de films : ', len(movies.movieId.unique()), '\n')
print(movies.info())
movies.head(10)

Nbre de films :  8113 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8113 entries, 23 to 27276
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  8113 non-null   object
 1   title    8113 non-null   object
 2   genres   8113 non-null   object
 3   year     8109 non-null   object
dtypes: object(4)
memory usage: 316.9+ KB
None


/tmp/ipykernel_22908/2090118895.py:32: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies['genres'] = movies['genres'].str.replace('|',' ')


,movieId,title,genres,year
23,24,Powder,Drama Sci-Fi,1995
31,32,Twelve Monkeys (a.k.a. 12 Monkeys),Mystery Sci-Fi Thriller,1995
42,43,Restoration,Drama,1995
50,51,Guardian Angel,Action Drama Thriller,1994
72,73,"Misérables, Les",Drama War,1995
85,86,White Squall,Action Adventure Drama,1996
112,114,Margaret's Museum,Drama,1995
113,115,Happiness Is in the Field (Bonheur est dans le...,Comedy,1995
156,158,Casper,Adventure Children,1995
179,181,Mighty Morphin Power Rangers: The Movie,Action Children,1995


In [18]:
# Load /  A ajouter : chargement direct dpuis url
# links = pd.read_csv('data_mdl/links.zip',compression='zip', sep=',')

# links = links.drop(['tmdbId'], axis=1)

# print('Nbre de films : ', len(movies.movieId.unique()), '\n')
# print('imdbId min : ',links.imdbId.min(), ' imdbId max : ', links.imdbId.max(), '\n')
# print(links.info())
# links.head(5)

## Merge

In [19]:
mdl_content = movies

#mdl_content = movies.merge(right=links, left_on='movieId', right_on='movieId', how='left')


mdl_content.head(5)

,movieId,title,genres
23,24,Powder,Drama Sci-Fi
31,32,Twelve Monkeys (a.k.a. 12 Monkeys),Mystery Sci-Fi Thriller
42,43,Restoration,Drama
50,51,Guardian Angel,Action Drama Thriller
72,73,"Misérables, Les",Drama War


## Save

In [20]:
mdl_content.to_csv("data_processed/mdl_content.csv.zip", index=False, compression="zip")